# CropGuard — train the pest model

Trains the pest branch of the Smart Farming Assistant on **Pestopia** (Indian pests)
and exports a bundle that runs on a field device with no internet.

**Runtime → Change runtime type → T4 GPU** before you start. Whole run is roughly
30–60 minutes depending on how large Pestopia turns out to be.

The notebook does not hide anything from you: it prints which classes were found,
which were **rejected**, and what the EDA says to change before the long run.


## 1. Check the GPU


In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — Runtime > Change runtime type > T4 GPU, then rerun.')
    print('Training on CPU will take many hours.')


## 2. Get the code

If the repo is private, either make it public, or use a token:
`!git clone https://<TOKEN>@github.com/omsatpute61-afk/Model.git`


In [ ]:
!git clone -b claude/smart-farming-edge-ai-2hzch8 https://github.com/omsatpute61-afk/Model.git
%cd Model
!git log --oneline -1


### Dependencies

Colab already ships torch built against its CUDA driver — **do not reinstall it**,
that is the usual way to break GPU support in a Colab notebook. Only the extras.


In [ ]:
!pip install -q kagglehub onnx onnxruntime pyyaml

import sys
sys.path.insert(0, '/content/Model/src')   # import cropguard without installing

import cropguard
from cropguard.taxonomy import load_taxonomy
print('cropguard', cropguard.__version__, '|', len(load_taxonomy()), 'classes in the taxonomy')


## 3. Download Pestopia

Public datasets usually download anonymously. If you get a 401/403, add your Kaggle
credentials: Colab left sidebar → 🔑 Secrets → add `KAGGLE_USERNAME` and `KAGGLE_KEY`
(from kaggle.com → Settings → API → Create New Token), then rerun this cell.


In [ ]:
import os

try:  # pick up Colab secrets if you added them
    from google.colab import userdata
    for key in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

import kagglehub
PEST_PATH = kagglehub.dataset_download('shruthisindhura/pestopia')
print('downloaded to:', PEST_PATH)


### What is actually in it?

Worth looking before training. This prints the real folder names and image counts —
the thing neither of us could see until now.


In [ ]:
from pathlib import Path
from collections import Counter

IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
root = Path(PEST_PATH)

# Descend past any single wrapper directory the archive may have.
while True:
    subs = [d for d in root.iterdir() if d.is_dir()]
    files = [f for f in root.iterdir() if f.suffix.lower() in IMG]
    if len(subs) == 1 and not files:
        root = subs[0]
    else:
        break

print('dataset root:', root, '\n')
counts = Counter()
for d in sorted(p for p in root.iterdir() if p.is_dir()):
    counts[d.name] = sum(1 for f in d.rglob('*') if f.suffix.lower() in IMG)

print(f'{len(counts)} class folders, {sum(counts.values()):,} images\n')
for name, n in counts.most_common():
    print(f'{n:>7,}  {name}')


### Which of those are diseases, not pests?

A pest dataset shipping fungal or bacterial classes is common, and letting them
into the pest branch is not harmless: the same condition ends up on **both**
branches of the model with labels from two sources, and the pest branch's
life-stage and economic-threshold logic gets applied to a fungus, which has
neither larvae nor a larvae-per-plant count.

The ingest rejects them automatically. This cell just shows you what it will do.


In [ ]:
from cropguard.data.ingest import looks_like_disease, check_source_kind

tax = load_taxonomy()
pests, rejects, unknown = [], [], []
for name, n in counts.items():
    cls = tax.resolve(name)
    if cls is not None:
        reason = check_source_kind(cls, 'pest', name)
        (rejects if reason else pests).append((name, n, reason or cls.id))
    elif looks_like_disease(name):
        rejects.append((name, n, 'name describes a plant disease, not an insect'))
    else:
        unknown.append((name, n, None))

print(f'PEST classes kept: {len(pests)}  ({sum(n for _, n, _ in pests):,} images)')
for name, n, cid in sorted(pests, key=lambda x: -x[1]):
    print(f'   {n:>7,}  {name}  ->  {cid}')

print(f'\nREJECTED as disease: {len(rejects)}  ({sum(n for _, n, _ in rejects):,} images)')
for name, n, why in sorted(rejects, key=lambda x: -x[1]):
    print(f'   {n:>7,}  {name}  ({why})')

print(f'\nUNMAPPED — need a taxonomy alias: {len(unknown)}  ({sum(n for _, n, _ in unknown):,} images)')
for name, n, _ in sorted(unknown, key=lambda x: -x[1]):
    print(f'   {n:>7,}  {name}')
print('\nSend the UNMAPPED list back and they can be added to the taxonomy.')


## 4. Ingest → clean → split → EDA

Removes unreadable files, duplicates (including the same image filed under two
different labels, which caps the accuracy any model can reach), near-duplicates
that would otherwise leak across the split, and classes too small to evaluate.

Nothing is deleted from the download — every dropped image is listed in `dropped.csv`.


In [ ]:
!python scripts/prepare_real_dataset.py \
    --pest "$PEST_PATH" \
    --out artifacts/data/pest \
    --min-per-class 40 \
    --workers 2


### Read the EDA before the long run

The recommendations at the top name settings in `configs/pest.yaml` to change.


In [ ]:
print(open('artifacts/data/pest/eda.md').read())


## 5. Train

Adjust `configs/pest.yaml` first if the EDA told you to — particularly
`image_size` (if most images are small) and `use_focal` (if the tail is mild).

Overrides work from the command line too, e.g. `--set data.image_size=160`.


In [ ]:
!python -m cropguard.train --config configs/pest.yaml


## 6. Evaluate

Reports macro F1 rather than accuracy — a long-tailed pest set gives a high
accuracy to a model that always answers with the commonest pest and misses every
outbreak. Also prints the **cross-category error rate** and the confusions that
would change what a farmer buys.


In [ ]:
!python -m cropguard.evaluate --run artifacts/runs/cropguard-pest --split test


## 7. Export the edge bundle

ONNX + INT8, each verified against torch, with the model card, taxonomy, advisories
and a novelty detector refitted per artefact.


In [ ]:
!python -m cropguard.export --run artifacts/runs/cropguard-pest --formats onnx,int8
!echo '--- bundle ---' && ls -la artifacts/runs/cropguard-pest/export/


### Speed and size on this machine

Colab is not a Raspberry Pi — treat these as relative numbers and re-run
`cropguard.benchmark` on the real board before promising a latency.


In [ ]:
!python -m cropguard.benchmark --bundle artifacts/runs/cropguard-pest/export --compare


## 8. Try the device runtime

This is what actually runs in the field: image in, diagnosis with calibrated
confidence out, plus the advisory a farmer would receive — including the
decision to **refuse to answer** on an image the model does not understand.


In [ ]:
import random, numpy as np
from PIL import Image
from cropguard.edge import EdgeClassifier
from cropguard.data.manifest import read_manifest

clf = EdgeClassifier('artifacts/runs/cropguard-pest/export')
print('backend:', clf.backend, '| model:', clf._model_path.name)
print('threshold:', clf.card.policy.min_confidence, '| classes:', len(clf.card.class_ids))

records = [r for r in read_manifest('artifacts/data/pest/manifest.csv') if r.split == 'test']
random.seed(0)
correct = 0
for rec in random.sample(records, min(8, len(records))):
    d = clf.diagnose(rec.path)
    correct += d.class_id == rec.class_id
    mark = 'ok ' if d.class_id == rec.class_id else '   '
    print(f'{mark}{rec.class_id:<28} -> {d.class_id:<28} p={d.confidence:.2f} {d.latency_ms:5.1f}ms')
    if d.accepted and d.advisory and d.advisory.urgency in ('warning', 'critical'):
        print(f'      SMS: {d.advisory.to_sms()}')

print(f'\n{correct}/8 correct on this sample')

print('\n--- an image the model should refuse ---')
noise = Image.fromarray(np.random.default_rng(0).integers(0, 255, (320, 320, 3)).astype('uint8'))
d = clf.diagnose(noise)
print('accepted:', d.accepted, '| confidence:', round(d.confidence, 3),
      '| novelty:', None if d.novelty is None else round(d.novelty))
print('reason:', d.reason or '(accepted)')
if d.advisory:
    print('advice:', d.advisory.message[:150])


## 9. Download the bundle

Everything the field device needs, in one zip — models, model card, taxonomy,
advisories and the novelty detectors. Grab it before the Colab session expires.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('cropguard_pest_bundle', 'zip', 'artifacts/runs/cropguard-pest/export')
shutil.make_archive('cropguard_pest_run', 'zip', 'artifacts/runs/cropguard-pest')
files.download('cropguard_pest_bundle.zip')   # ~2 MB, what the device runs
# files.download('cropguard_pest_run.zip')    # + checkpoints, history, eval reports


---

### If accuracy is short

1. **Read `eda.md` again.** Most of the time the answer is there: too few images in
   the weak classes, a resolution mismatch, or an imbalance that needs focal loss.
2. `--set data.image_size=160` if the EDA says most images are small — upscaling
   invents detail that is not in the file.
3. `--set optim.epochs=60` — 30 is conservative for a long-tailed set.
4. `--set model.backbone=efficientnet_b0` if the device budget allows it, then
   re-benchmark on the real board.
5. `--set optim.mixup_alpha=0.2` if the rare classes overfit.

### What the numbers mean

**Macro F1** is the headline, not accuracy. **Per-class recall** on the pests that
matter is what decides whether an outbreak gets caught. **Cross-category error** is
the one that costs money — it means the farmer buys the wrong input.

A high score on a curated corpus still says little about a phone photo taken at noon
in a standing crop. Hold back a few hundred genuinely field-shot images as a second
test set; that number is the one worth reporting.
